In [2]:
import pandas as pd
import numpy as np
from datetime import timedelta, date

df = pd.read_csv("datasets/BE_Data_UTC.csv")
df["Date"] = pd.to_datetime(df["Date"])
df["date_only"] = df["Date"].dt.date
df["hour"] = df["Date"].dt.hour
df = df.sort_values("Date").reset_index(drop=True)

def naive_forecast(forecast_date, df):
    if forecast_date.weekday() in (1, 2, 3, 4):
        lag_days = 1
    else:
        lag_days = 7
    source_date = forecast_date - timedelta(days=lag_days)
    source_rows = df[df["date_only"] == source_date].sort_values("hour")
    if len(source_rows) != 24:
        return None, None
    return source_rows["Price"].values, source_date

forecast, source_date = naive_forecast(date(2023, 6, 1), df)
print(forecast[:5])
print(source_date)

[74.2  62.64 61.61 72.7  77.64]
2023-05-31


In [3]:
import pandas as pd
from datetime import timedelta, date

df = pd.read_csv("datasets/BE_Data_UTC.csv")
df["Date"] = pd.to_datetime(df["Date"])
df["date_only"] = df["Date"].dt.date

test_date = date(2023, 6, 1)  # any date you're testing in the dashboard
day = df[df["date_only"] == test_date].sort_values("Date")
print(day[["Price", "Price_CH", "Price_DE_LU_15min", "Price_AT_15min"]].head())

       Price  Price_CH  Price_DE_LU_15min  Price_AT_15min
21888  70.10     69.60             72.885         66.1800
21889  65.05     66.49             71.535         67.0200
21890  67.40     67.89             71.400         67.8775
21891  75.39     74.55             82.015         78.1025
21892  88.28     86.15             96.495         91.8275


In [1]:
import sqlite3

# 1. Connect (creates the file if it doesn't exist yet)
conn = sqlite3.connect("test_practice.db")

# 2. Create a table
conn.execute("""
    CREATE TABLE IF NOT EXISTS notes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        content TEXT NOT NULL
    )
""")
conn.commit()

In [2]:
conn.execute("INSERT INTO notes (content) VALUES (?)", ("my first note",))
conn.execute("INSERT INTO notes (content) VALUES (?)", ("second note",))
conn.commit()

In [3]:
cursor = conn.execute("SELECT * FROM notes")
rows = cursor.fetchall()
print(rows)

[(1, 'my first note'), (2, 'second note')]


In [4]:
cursor = conn.execute("SELECT * FROM notes WHERE id = ?", (1,))
print(cursor.fetchone())  # fetchone() instead of fetchall() when you expect at most one row

(1, 'my first note')


In [5]:
conn.execute("UPDATE notes SET content = ? WHERE id = ?", ("edited note", 1))
conn.execute("DELETE FROM notes WHERE id = ?", (2,))
conn.commit()

In [6]:
conn.close()

In [7]:
import sys
# sys.path.append(".")  # only needed if the notebook isn't already in the same folder as db.py

from db import load_feedback, load_users

feedback_df = load_feedback()
feedback_df.head(20)

,id,expert_id,forecast_date,hour,forecast,adjusted,flagged,price_ch,price_de_lu,price_at,timestamp
0,1,expert_1,2025-01-29,0,32.86,100.00,True,117.57,57.4725,81.1750,2026-08-13T00:34:01.388034+00:00
1,2,expert_1,2025-01-29,1,23.75,100.00,True,113.78,53.8900,80.2875,2026-08-13T00:34:01.388034+00:00
2,3,expert_1,2025-01-29,2,28.79,28.79,False,113.07,57.4175,79.5750,2026-08-13T00:34:01.388034+00:00
3,4,expert_1,2025-01-29,3,29.70,29.70,False,116.14,73.0050,86.4600,2026-08-13T00:34:01.388034+00:00
4,5,expert_1,2025-01-29,4,28.78,28.78,True,124.31,85.6025,108.4400,2026-08-13T00:34:01.388034+00:00
5,6,expert_1,2025-01-29,5,47.24,47.24,False,138.94,101.9875,128.9750,2026-08-13T00:34:01.388034+00:00
6,7,expert_1,2025-01-29,6,72.06,72.06,False,148.26,128.6500,149.7025,2026-08-13T00:34:01.388034+00:00
7,8,expert_1,2025-01-29,7,94.31,94.31,False,146.84,137.4750,155.8875,2026-08-13T00:34:01.388034+00:00
8,9,expert_1,2025-01-29,8,87.62,87.62,False,144.05,129.8800,137.7575,2026-08-13T00:34:01.388034+00:00
9,10,expert_1,2025-01-29,9,91.20,91.20,False,143.06,121.5675,131.5300,2026-08-13T00:34:01.388034+00:00


In [8]:
feedback_df.describe()
feedback_df[feedback_df["expert_id"] == "expert_1"]
feedback_df.groupby("expert_id").size()

expert_id
expert_1    24
expert_2    24
dtype: int64

In [9]:
users = load_users()
users   # a dictionary

{'irinalzr1': {'email': 'irinalzr1@gmail.com',
  'password': '$2b$12$hKy6DoQg4Rd4uNXCL11mwedjxZadaMzhZT8F/gJ7SXK.JfczypDA6',
  'role': 'admin'},
 'expert_1': {'email': 'expert1@kuleuven.be',
  'password': '$2b$12$NyV1YOsB3Jva9sXHJTRkVuOux/5yJwGXawZqKH3SHnkAtJfB9Iu2i',
  'role': 'expert'},
 'expert_2': {'email': 'expert2@kuleuven.be',
  'password': '$2b$12$mb4Qr7/rWhUxxG/tHA9uz.pXaYWYiogW1L2bUDNzkDzei.GUDZiFm',
  'role': 'expert'}}

In [10]:
import pandas as pd
pd.DataFrame(users).T   # .T flips rows/columns since the dict is naturally keyed by username

,email,password,role
irinalzr1,irinalzr1@gmail.com,$2b$12$hKy6DoQg4Rd4uNXCL11mwedjxZadaMzhZT8F/gJ...,admin
expert_1,expert1@kuleuven.be,$2b$12$NyV1YOsB3Jva9sXHJTRkVuOux/5yJwGXawZqKH3...,expert
expert_2,expert2@kuleuven.be,$2b$12$mb4Qr7/rWhUxxG/tHA9uz.pXaYWYiogW1L2bUDN...,expert
